## Evaluation 

- newly labelled dataset

In [1]:
import pandas as pd

In [2]:
eval_data = pd.read_csv("s3://open-jobs-lake/job_quality/outputs/evaluation/evaluation_data_12_08_24_per_sentence_evaluation_14_08.csv")

In [3]:
eval_data.head(2)

,id,company_raw,job_title_raw,job_location_raw,created,type,sector,parent_sector,knowledge_domain,occupation,...,DISABILITY,HEALTH,M_HEALTH,SPONSORSHIP,REWARD,MISC,AUTONOMY,SENSE OF PURPOSE,SOCIAL,VOICE REPRESENTATION
0,41721516,Servoca Education Resourcing,Early Years TA with SEN experience needed in B...,"London, South East England",2021-01-11 00:00:00,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41721516,Servoca Education Resourcing,Early Years TA with SEN experience needed in B...,"London, South East England",2021-01-11 00:00:00,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
eval_data.columns

Index(['id', 'company_raw', 'job_title_raw', 'job_location_raw', 'created',
       'type', 'sector', 'parent_sector', 'knowledge_domain', 'occupation',
       'description', 'itl_3_code', 'itl_3_name', 'origin',
       'clean_description', 'sentences', 'Labelled by?', 'L&D', 'CAREER',
       'HOURS', 'FLEX_HOURS', 'SHIFT', 'LOC', 'FLEX_LOC', 'CONTRACT', 'LEAVE',
       'COMP', 'PERKS', 'CARING', 'DISABILITY', 'HEALTH', 'M_HEALTH',
       'SPONSORSHIP', 'REWARD', 'MISC', 'AUTONOMY', 'SENSE OF PURPOSE',
       'SOCIAL', 'VOICE REPRESENTATION'],
      dtype='object')

In [5]:
usable_eval_data = eval_data[pd.notnull(eval_data['Labelled by?'])]
len(usable_eval_data)

1446

In [6]:
jq_cols = ['L&D', 'CAREER',
       'HOURS', 'FLEX_HOURS', 'SHIFT', 'LOC', 'FLEX_LOC', 'CONTRACT', 'LEAVE',
       'COMP', 'PERKS', 'CARING', 'DISABILITY', 'HEALTH', 'M_HEALTH',
       'SPONSORSHIP', 'REWARD', 'MISC', 'AUTONOMY', 'SENSE OF PURPOSE',
       'SOCIAL', 'VOICE REPRESENTATION']

In [7]:
usable_eval_data.columns

Index(['id', 'company_raw', 'job_title_raw', 'job_location_raw', 'created',
       'type', 'sector', 'parent_sector', 'knowledge_domain', 'occupation',
       'description', 'itl_3_code', 'itl_3_name', 'origin',
       'clean_description', 'sentences', 'Labelled by?', 'L&D', 'CAREER',
       'HOURS', 'FLEX_HOURS', 'SHIFT', 'LOC', 'FLEX_LOC', 'CONTRACT', 'LEAVE',
       'COMP', 'PERKS', 'CARING', 'DISABILITY', 'HEALTH', 'M_HEALTH',
       'SPONSORSHIP', 'REWARD', 'MISC', 'AUTONOMY', 'SENSE OF PURPOSE',
       'SOCIAL', 'VOICE REPRESENTATION'],
      dtype='object')

In [8]:
per_job_ad = usable_eval_data.groupby('id')[jq_cols].sum().reset_index()

per_job_ad['all_jq'] = per_job_ad[jq_cols].apply(lambda x: list(set([jq_name for jq_name in jq_cols if x[jq_name]!=0])), axis=1)

In [9]:
eval_dict = dict(zip(per_job_ad['id'], per_job_ad['all_jq']))

# Predict job quality for evaluation dataset


In [10]:
mapping_evaluation_dir = "s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/mapping_evaluation"

keyword_lookup_data = pd.read_csv(
    f"{mapping_evaluation_dir}/keyword_lookup_v7_080824.csv"
)
tp_to_subcategory = dict(
    zip(keyword_lookup_data["target_phrase"], keyword_lookup_data["subcategory"])
)
tp_to_dimension = dict(
    zip(keyword_lookup_data["target_phrase"], keyword_lookup_data["dimension"])
)
subcategory_to_dimension = dict(
    zip(keyword_lookup_data["subcategory"], keyword_lookup_data["dimension"])
)

In [11]:
from dap_job_quality.pipeline.find_job_quality import JobQuality

2024-08-14 17:33:13,464 - datasets - INFO - PyTorch version 2.2.2 available.
2024-08-14 17:33:13,466 - datasets - INFO - Polars version 0.20.31 available.
2024-08-14 17:33:16,940 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [12]:
job_quality = JobQuality()
job_quality.load()

eval_job_adverts = (
    usable_eval_data[["id", "description"]].drop_duplicates().reset_index(drop=True)
)

2024-08-14 17:33:17,089 - root - INFO - Loading models and variables
2024-08-14 17:33:17,333 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/elizabethgallagher/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/elizabethgallagher/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-08-14 17:33:19,189 - root - INFO - Downloading the model...
2024-08-14 17:33:34,862 - root - INFO - Loading the model and tokenizer...


/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


2024-08-14 17:33:35,401 - root - INFO - Calculating embeddings for 115 target phrases ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [13]:
len(eval_job_adverts)

66

In [14]:
jq_df_filtered, _ = job_quality.extract_job_quality(
            eval_job_adverts, "id", "description"
)
jq_predictions = jq_df_filtered[
    ["id", "target_phrase", "cosine_similarity"]
]

2024-08-14 17:33:44,782 - root - INFO - Predicting job quality sentences for 1883 sentences ...
Time taken: 95.02 seconds
2024-08-14 17:35:19,827 - root - INFO - Calculating embeddings for 4075 ngrams ...


Batches:   0%|          | 0/128 [00:00<?, ?it/s]

Time taken: 11.84 seconds


In [15]:
jq_predictions["subcategory"] = jq_predictions["target_phrase"].map(
    tp_to_subcategory
)
jq_predictions["dimension"] = jq_predictions["target_phrase"].map(tp_to_dimension)

/var/folders/xc/s255_bsx0l7cbx43t290kjtr0000gn/T/ipykernel_14208/2609569756.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jq_predictions["subcategory"] = jq_predictions["target_phrase"].map(
/var/folders/xc/s255_bsx0l7cbx43t290kjtr0000gn/T/ipykernel_14208/2609569756.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jq_predictions["dimension"] = jq_predictions["target_phrase"].map(tp_to_dimension)


In [16]:
jq_predictions.head(2)

,id,target_phrase,cosine_similarity,subcategory,dimension
0,41757472,holiday pay,0.674074,LEAVE,pay and benefits
1,45379079,benefits,0.803152,PERKS,pay and benefits


In [17]:
per_job_ad_preds = pd.get_dummies(jq_predictions[['id', 'subcategory']], columns=['subcategory'], prefix='prediction').groupby('id').sum().reset_index()
per_job_ad_preds.head(2)

,id,prediction_AUTONOMY,prediction_CAREER,prediction_CARING,prediction_COMP,prediction_CONTRACT,prediction_DISABILITY,prediction_FLEX_HOURS,prediction_FLEX_LOC,prediction_HEALTH,...,prediction_LEAVE,prediction_LOC,prediction_MISC,prediction_M_HEALTH,prediction_PERKS,prediction_REWARD,prediction_SENSE OF PURPOSE,prediction_SHIFT,prediction_SOCIAL,prediction_SPONSORSHIP
0,41721516,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,41757472,0,2,0,0,0,0,2,0,0,...,2,0,0,0,2,0,0,0,0,0


In [18]:
per_job_ad.head(2)

,id,L&D,CAREER,HOURS,FLEX_HOURS,SHIFT,LOC,FLEX_LOC,CONTRACT,LEAVE,...,HEALTH,M_HEALTH,SPONSORSHIP,REWARD,MISC,AUTONOMY,SENSE OF PURPOSE,SOCIAL,VOICE REPRESENTATION,all_jq
0,41721516,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[COMP, LOC]"
1,41757472,0.0,3.0,0.0,2.0,0.0,0.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[FLEX_HOURS, FLEX_LOC, LEAVE, PERKS, CAREER, C..."


In [19]:
comparison_df = per_job_ad.merge(per_job_ad_preds, on='id', how='left')
comparison_df.head(2)

,id,L&D,CAREER,HOURS,FLEX_HOURS,SHIFT,LOC,FLEX_LOC,CONTRACT,LEAVE,...,prediction_LEAVE,prediction_LOC,prediction_MISC,prediction_M_HEALTH,prediction_PERKS,prediction_REWARD,prediction_SENSE OF PURPOSE,prediction_SHIFT,prediction_SOCIAL,prediction_SPONSORSHIP
0,41721516,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,41757472,0.0,3.0,0.0,2.0,0.0,0.0,2.0,1.0,1.0,...,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0


## Evaluate

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

In [23]:
class_rep_per_jq = {}
for jq_measure in jq_cols:
    if f"prediction_{jq_measure}" in comparison_df:
        pred_list = comparison_df[f"prediction_{jq_measure}"]!=0
    else:
        pred_list = [False]*len(comparison_df)
        
    class_rep = classification_report(
        comparison_df[jq_measure]!=0,
        pred_list,
        output_dict=True
    )
    class_rep_per_jq[jq_measure] = class_rep

/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/elizabethgallagher/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _wa

In [25]:
{k: v['weighted avg'] for k, v in class_rep_per_jq.items()}

{'L&D': {'precision': 0.7913242396001017,
  'recall': 0.7878787878787878,
  'f1-score': 0.7888755980861243,
  'support': 66.0},
 'CAREER': {'precision': 0.8045469865490088,
  'recall': 0.7878787878787878,
  'f1-score': 0.7932659932659932,
  'support': 66.0},
 'HOURS': {'precision': 0.8821548821548821,
  'recall': 0.8484848484848485,
  'f1-score': 0.8433857808857809,
  'support': 66.0},
 'FLEX_HOURS': {'precision': 0.7582023239917978,
  'recall': 0.696969696969697,
  'f1-score': 0.7122897325121794,
  'support': 66.0},
 'SHIFT': {'precision': 0.7631964809384165,
  'recall': 0.8181818181818182,
  'f1-score': 0.7838432753686991,
  'support': 66.0},
 'LOC': {'precision': 0.5737373737373738,
  'recall': 0.6060606060606061,
  'f1-score': 0.5813566799009128,
  'support': 66.0},
 'FLEX_LOC': {'precision': 0.7105436935945411,
  'recall': 0.7424242424242424,
  'f1-score': 0.6989974511469839,
  'support': 66.0},
 'CONTRACT': {'precision': 0.7537730243612597,
  'recall': 0.7575757575757576,
  'f1-s

## When is it incorrect?

In [47]:
comparison_df

,id,L&D,CAREER,HOURS,FLEX_HOURS,SHIFT,LOC,FLEX_LOC,CONTRACT,LEAVE,...,prediction_LEAVE,prediction_LOC,prediction_MISC,prediction_M_HEALTH,prediction_PERKS,prediction_REWARD,prediction_SENSE OF PURPOSE,prediction_SHIFT,prediction_SOCIAL,prediction_SPONSORSHIP
0,41721516,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,41757472,0.0,3.0,0.0,2.0,0.0,0.0,2.0,1.0,1.0,...,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
2,41878168,2.0,0.0,1.0,3.0,0.0,1.0,2.0,0.0,2.0,...,4.0,1.0,2.0,0.0,9.0,0.0,0.0,0.0,1.0,0.0
3,41991021,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,42038449,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,48251662,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
62,48310356,2.0,1.0,3.0,1.0,0.0,1.0,1.0,0.0,2.0,...,10.0,0.0,1.0,2.0,5.0,0.0,1.0,0.0,2.0,0.0
63,48330370,4.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
64,48472322,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0


In [100]:
jq_measure = 'CONTRACT'
incorrect_jq = comparison_df[(comparison_df[f"prediction_{jq_measure}"]!=0) != (comparison_df[jq_measure]!=0)][[
    "id", jq_measure, f"prediction_{jq_measure}"]].merge(eval_job_adverts, how='left', on='id')
#.to_csv(f"eval_incorrect_{jq_measure}.csv")
print(len(incorrect_jq))
i = 3
print(incorrect_jq.iloc[i])
print(incorrect_jq.iloc[i]['description'])

16
id                                                              42419334
CONTRACT                                                             1.0
prediction_CONTRACT                                                  0.0
description            [ PPA Teacher required for Ofsted Good Primary...
Name: 3, dtype: object
[ PPA Teacher required for Ofsted Good Primary School in Berkshire - QTS/NQT Full time Long term postSeptember 2021 start date Are you a Primary school teacher with the drive and creativity to make a positive impact in a school? We’re looking for PPA teachers in Berkshire to do just that while avoiding the responsibilities of being a full-time class teacher. About the role If successful, you’ll be joining a popular school based in Berkshire. Resources are of an exceptional standard and the school offers an additional benefits package to all staff members. Staff are dedicated to improving the attainment and progress of their students with a key focus on the community. The sc

In [37]:
eval_job_adverts.iloc[pred_list[pred_list!=truth_list].index]

,id,description
1,41757472,[ Are you an ambitious ACA or ACCA seeking a n...
7,42630006,[ CNC Miller (Days) 3 Month Contract Cambridge...
9,42758914,[ Warehouse Operative - Mid Shift VacanciesBas...
10,42852730,[ Sales Executive Red Bull (working for REL Fi...
16,43535813,[ Are you an experienced Business Development ...
19,44017518,[We are looking for Registered Nurses to join ...
27,45561009,[ Job; Digital Solutions Sales Consultant - Re...
34,47217245,[* Top 25 PR Agency* Hybrid Working* Amazing b...
37,47896751,[ We have an exciting opportunity for an ambit...
39,47965016,"[ Our client is a 13-year-old male, who is tot..."


In [29]:
truth_list


0     False
1     False
2      True
3     False
4     False
      ...  
61     True
62     True
63     True
64    False
65    False
Name: L&D, Length: 66, dtype: bool